# Brain Stroke Classification

A simple walkthrough of stroke prediction with decision trees, starting from a baseline model and then trying a few small improvements.


## Import and setup

Imports, plotting settings, and shared constants are set up here.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = Path("brain_stroke.csv")

print(f"Data file exists: {DATA_PATH.exists()}")


## Load the data

The dataset is loaded here, followed by a quick first look at its size and columns.


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Kích thước dữ liệu:", df.shape)
print("Các cột:", list(df.columns))
print("\n5 dòng đầu tiên:")
display(df.head())


In [ ]:
print("Thông tin kiểu dữ liệu:")
df.info()

print("\nThống kê mô tả cho các cột số:")
display(df.describe().T)


## Data check

This section checks for missing values, duplicate rows, and class imbalance in `stroke`.


In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
duplicate_count = df.duplicated().sum()
class_distribution = df["stroke"].value_counts().sort_index()
class_ratio = (df["stroke"].value_counts(normalize=True).sort_index() * 100).round(2)

print("Số lượng giá trị thiếu theo từng cột:")
display(missing_summary.to_frame(name="missing_count"))

print(f"Số dòng bị trùng: {duplicate_count}")

print("\nPhân bố biến mục tiêu `stroke`:")
summary_df = pd.DataFrame({
    "count": class_distribution,
    "ratio_percent": class_ratio,
})
display(summary_df)


## Quick plots

These plots give a quick feel for the target distribution and the spread of age, glucose, and BMI.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# A quick look at the main distributions in the dataset.
sns.countplot(data=df, x="stroke", palette="Set2", ax=axes[0, 0])
axes[0, 0].set_title("Stroke class distribution")
axes[0, 0].set_xlabel("Stroke")
axes[0, 0].set_ylabel("Number of samples")

sns.histplot(data=df, x="age", hue="stroke", bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Age distribution by stroke")

sns.histplot(data=df, x="avg_glucose_level", hue="stroke", bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Glucose distribution by stroke")

sns.histplot(data=df, x="bmi", hue="stroke", bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("BMI distribution by stroke")

plt.tight_layout()
plt.show()


## Preprocessing

Missing BMI values are filled, categorical features are encoded, and the data is split into train and test sets.


In [ ]:
df_model = df.copy()

# Fill missing BMI values with the mean.
if df_model["bmi"].isna().sum() > 0:
    df_model["bmi"] = df_model["bmi"].fillna(df_model["bmi"].mean())

# Turn categorical columns into numeric features.
categorical_columns = df_model.select_dtypes(include="object").columns.tolist()
print("Categorical columns to encode:", categorical_columns)

df_encoded = pd.get_dummies(df_model, columns=categorical_columns, drop_first=True)

# Separate features and target.
X = df_encoded.drop(columns=["stroke"])
y = df_encoded["stroke"]

# Keep the class ratio similar in both train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("Stroke rate in train:", round(y_train.mean() * 100, 2), "%")
print("Stroke rate in test :", round(y_test.mean() * 100, 2), "%")


## Helper functions

A few helper functions are grouped here to keep the later cells cleaner.


In [ ]:
def evaluate_tree_model(model_name, model, X_train, X_test, y_train, y_test):
    # Train model.
    model.fit(X_train, y_train)

    # Predict labels and probabilities on the test set.
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Store the main metrics together for easier comparison.
    result = {
        "model_name": model_name,
        "model": model,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "accuracy": accuracy_score(y_test, y_pred),
        "error_rate": 1 - accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "depth": model.get_depth(),
        "n_leaves": model.get_n_leaves(),
    }
    return result


def show_confusion_matrix(cm, title):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Stroke", "Stroke"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(title)
    plt.grid(False)
    plt.show()


def summarize_overfitting(train_acc, test_acc):
    gap = train_acc - test_acc
    if gap >= 0.10:
        return f"Train accuracy is higher than test accuracy by {gap:.4f}, which suggests clear overfitting."
    if gap >= 0.05:
        return f"The train/test gap is {gap:.4f}, so there may be mild overfitting."
    return f"The train/test gap is {gap:.4f}, so overfitting does not look too severe."


def result_to_frame(result):
    return pd.DataFrame(
        {
            "Metric": ["Train Accuracy", "Accuracy", "Error Rate", "Precision", "Recall", "F1-Score", "ROC-AUC", "Tree Depth", "Number of Leaves"],
            "Value": [
                result["train_accuracy"],
                result["accuracy"],
                result["error_rate"],
                result["precision"],
                result["recall"],
                result["f1_score"],
                result["roc_auc"],
                result["depth"],
                result["n_leaves"],
            ],
        }
    )


## Baseline tree

The default decision tree is trained here as a baseline for comparison.


In [ ]:
baseline_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_result = evaluate_tree_model(
    "Baseline Decision Tree",
    baseline_model,
    X_train,
    X_test,
    y_train,
    y_test,
)

print("=== BASELINE MODEL RESULTS ===")
display(result_to_frame(baseline_result))

print("Classification report:")
print(classification_report(y_test, baseline_result["y_pred"], zero_division=0))


In [ ]:
# Show the confusion matrix and the top part of the tree.
show_confusion_matrix(
    baseline_result["confusion_matrix"],
    "Confusion Matrix - Baseline Decision Tree"
)

plt.figure(figsize=(24, 12))
plot_tree(
    baseline_result["model"],
    feature_names=X.columns,
    class_names=["No Stroke", "Stroke"],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3,
)
plt.title("Baseline Decision Tree - Top 3 Levels")
plt.show()


## Look at the baseline tree

This part inspects the baseline tree through its depth, split rules, and feature importance.


In [ ]:
baseline_tree = baseline_result["model"]
root_feature_index = baseline_tree.tree_.feature[0]
root_threshold = baseline_tree.tree_.threshold[0]
root_feature_name = X.columns[root_feature_index]

feature_importance = (
    pd.Series(baseline_tree.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)

print("=== BASELINE TREE ANALYSIS ===")
print(f"Tree depth: {baseline_result['depth']}")
print(f"Number of leaves: {baseline_result['n_leaves']}")
print(f"Root split: {root_feature_name} <= {root_threshold:.2f}")
print(summarize_overfitting(baseline_result["train_accuracy"], baseline_result["accuracy"]))

print("\nTop 10 most important features:")
display(feature_importance.to_frame(name="importance"))

print("\nSimplified split rules for the first 4 levels:")
print(export_text(baseline_tree, feature_names=list(X.columns), max_depth=4))


In [ ]:
print("=== Quick takeaways ===")

top3 = feature_importance.head(3).index.tolist()

print(f"The baseline tree reaches a depth of {baseline_result['depth']} with {baseline_result['n_leaves']} leaves.")
print("Train accuracy is noticeably higher than test accuracy, so overfitting is worth watching.")
print(f"The three most influential features here are: {', '.join(top3)}.")
print("Near the top of the tree, age, glucose, and BMI appear early in the split rules.")
print("Since the classes are imbalanced, high accuracy does not automatically mean many stroke cases are being caught.")


## Try a few variations

A few lightweight variations are tested here, including class weighting, pruning, and entropy.


In [ ]:
model_candidates = {
    "Baseline": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Class Weight": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Pruning": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=10,
    ),
    "Entropy + Pruning": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        criterion="entropy",
        max_depth=5,
        min_samples_leaf=10,
    ),
}

all_results = {}

for model_name, model in model_candidates.items():
    all_results[model_name] = evaluate_tree_model(
        model_name,
        model,
        X_train,
        X_test,
        y_train,
        y_test,
    )

comparison_rows = []
for model_name, result in all_results.items():
    comparison_rows.append(
        {
            "Model": model_name,
            "Train Accuracy": result["train_accuracy"],
            "Accuracy": result["accuracy"],
            "Error Rate": result["error_rate"],
            "Precision": result["precision"],
            "Recall": result["recall"],
            "F1-Score": result["f1_score"],
            "ROC-AUC": result["roc_auc"],
            "Depth": result["depth"],
            "Leaves": result["n_leaves"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values(by="Accuracy", ascending=False)
display(comparison_df.round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.barplot(data=comparison_df, x="Accuracy", y="Model", palette="viridis", ax=axes[0, 0])
axes[0, 0].set_title("Accuracy comparison")

sns.barplot(data=comparison_df, x="Recall", y="Model", palette="magma", ax=axes[0, 1])
axes[0, 1].set_title("Recall comparison")

sns.barplot(data=comparison_df, x="F1-Score", y="Model", palette="crest", ax=axes[1, 0])
axes[1, 0].set_title("F1-score comparison")

sns.barplot(data=comparison_df, x="ROC-AUC", y="Model", palette="rocket", ax=axes[1, 1])
axes[1, 1].set_title("ROC-AUC comparison")

plt.tight_layout()
plt.show()


In [ ]:
best_accuracy_model = comparison_df.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
best_f1_model = comparison_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Model"]
best_recall_model = comparison_df.sort_values(by="Recall", ascending=False).iloc[0]["Model"]

print("=== Quick comparison ===")
print(f"Highest accuracy: {best_accuracy_model}")
print(f"Highest F1-score: {best_f1_model}")
print(f"Highest recall: {best_recall_model}")
print("")
print("If the goal is to catch more stroke cases, recall and ROC-AUC deserve more attention than accuracy alone.")


## Plot the improved tree

The improved tree is plotted here for an easier visual comparison.


In [ ]:
improved_tree_result = all_results["Entropy + Pruning"]

show_confusion_matrix(
    improved_tree_result["confusion_matrix"],
    "Confusion Matrix - Entropy + Pruning"
)

plt.figure(figsize=(22, 10))
plot_tree(
    improved_tree_result["model"],
    feature_names=X.columns,
    class_names=["No Stroke", "Stroke"],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3,
)
plt.title("Improved Decision Tree - Entropy + Pruning (Top 3 Levels)")
plt.show()


## Short note

Because the classes are imbalanced, accuracy alone can be misleading. Recall and F1-score matter too.
